# 🎨 GÁN NHÃN + ĐẾM — NGƯỜI & PHƯƠNG TIỆN (YOLOv8 + supervision)

**2 bài toán, MỖI bài 2 VIDEO**: 🚶 NGƯỜI (sảnh ga + lối đi bộ) · 🚗 PHƯƠNG TIỆN (giao lộ + cao tốc).
Mỗi video làm **cùng lúc**:
- 🎨 **MÀU THEO LỚP** — car/truck/bus mỗi loại 1 màu; người tất cả 1 màu (`ColorLookup.CLASS`).
- 🏷️ **TÊN NHÃN** — `car #3`, `truck #7`, `person #12` (loại + track-id).
- 🌀 **QUỸ ĐẠO** — vệt di chuyển (`TraceAnnotator`).
- 📏 **ĐẾM QUA VẠCH** (`LineZone`) — vào/ra.
- 🟩 **ĐẾM TRONG VÙNG** (`PolygonZone`) — số vật đang trong vùng / đỉnh.

> ⚠️ ĐẾM NGƯỜI trước đây kém vì dùng `market-square` (quay TỪ TRÊN CAO, người quá nhỏ →
> YOLO bỏ sót gần hết). Bản này đổi sang video NGƯỜI ĐI TRONG SẢNH (subway) + lối đi bộ
> (people-walking) — người TO, rõ, ngang tầm → YOLO bắt tốt hơn HẲN.

Xuất **video + ảnh** cho cả 4 video. Cần **GPU T4**.

## 1) Cài đặt + tải code

In [ ]:
import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
REPO = os.path.join(WORK, "VisionOS"); BR = "claude/locate-anything-test-suite-xwju2f"
if not os.path.isdir(os.path.join(REPO, ".git")):
    os.system(f"git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git {REPO}")
os.chdir(REPO)
os.system(f"git fetch -q origin {BR} && git checkout -q {BR} && git reset --hard -q origin/{BR}")
os.chdir(os.path.join(REPO, "VisionOS"))
os.system("pip install -q ultralytics 'supervision>=0.21' opencv-python-headless")
import torch
print("📁", os.getcwd(), "| 🖥️", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "❌ CHƯA BẬT GPU (Colab: Runtime→T4 · Kaggle: Settings→Accelerator→GPU)")

## 2) Nạp YOLOv8 (recall cao) + ANNOTATOR (màu theo LỚP)

In [ ]:
import sys, subprocess, warnings
warnings.simplefilter("ignore")
import numpy as np, cv2, supervision as sv
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
from recognition.detectors import load_standard_detector
from recognition.sv_counting import _to_sv, _make_polygon_zone   # class_id ỔN ĐỊNH → màu theo lớp

det = load_standard_detector(backend="ultralytics")   # YOLOv8x, imgsz1280, conf0.15
det.load()

CLASS = sv.ColorLookup.CLASS
box_ann   = sv.RoundBoxAnnotator(color_lookup=CLASS, thickness=2)
label_ann = sv.LabelAnnotator(color_lookup=CLASS, text_scale=0.5, text_thickness=1,
                              text_position=sv.Position.TOP_LEFT)
trace_ann = sv.TraceAnnotator(color_lookup=CLASS, thickness=3, trace_length=64)
print("✅ YOLO + annotator (màu theo LỚP) sẵn sàng.")

## 3) Hàm: gán nhãn (màu/tên/quỹ đạo) + ĐẾM VẠCH + ĐẾM VÙNG (1 lượt)

In [ ]:
def new_tracker():
    for kw in (dict(track_activation_threshold=0.1, minimum_consecutive_frames=1, lost_track_buffer=120),
               dict(track_thresh=0.1), {}):
        try: return sv.ByteTrack(**kw)
        except TypeError: continue
    return sv.ByteTrack()

def _px(pt, w, h): return (int(pt[0] / 100 * w), int(pt[1] / 100 * h))

def run_task(name, video, prompt, line_pct, zone_pct, out_mp4, reso=(1280, 720), max_frames=180):
    w, h = reso
    tr = new_tracker()
    try: sm = sv.DetectionsSmoother(length=8)
    except Exception: sm = None
    (sx, sy), (ex, ey) = _px(line_pct[:2], w, h), _px(line_pct[2:], w, h)
    line = sv.LineZone(start=sv.Point(sx, sy), end=sv.Point(ex, ey))     # ĐẾM VẠCH
    zpoly = np.array([_px(p, w, h) for p in zone_pct], dtype=np.int32)
    zone = _make_polygon_zone(sv, zpoly, w, h)                            # ĐẾM VÙNG
    vw = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"), 15, reso)
    cap = cv2.VideoCapture(video)
    i, seen, z_peak, by_class = 0, set(), 0, {}
    while i < max_frames:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, reso)
        sd = _to_sv(det.detect(fr, prompt).detections, sv, np)
        sd = tr.update_with_detections(sd)
        if sm is not None:
            try: sd = sm.update_with_detections(sd)
            except Exception: pass
        line.trigger(sd)                                                 # supervision đếm cắt vạch
        inz = np.asarray(zone.trigger(sd), dtype=bool) if len(sd) else np.zeros(0, bool)
        z_cur = int(inz.sum()); z_peak = max(z_peak, z_cur)
        names = sd.data.get("class_name") if getattr(sd, "data", None) else None
        if sd.tracker_id is not None:
            for j, t in enumerate(sd.tracker_id):
                if t is not None:
                    seen.add(int(t))
                    if names is not None: by_class[str(names[j])] = by_class.get(str(names[j]), 0) + 1
        # VẼ: quỹ đạo + box màu-theo-lớp + nhãn
        f = trace_ann.annotate(fr.copy(), sd)
        f = box_ann.annotate(f, sd)
        labels = [f"{(names[j] if names is not None else 'obj')} #{int(sd.tracker_id[j])}"
                  for j in range(len(sd))] if sd.tracker_id is not None else []
        f = label_ann.annotate(f, sd, labels=labels) if labels else f
        # VÙNG (xanh) + VẠCH (vàng) + banner số đếm
        ov = f.copy(); cv2.fillPoly(ov, [zpoly], (0, 170, 0)); cv2.addWeighted(ov, 0.2, f, 0.8, 0, f)
        cv2.polylines(f, [zpoly], True, (0, 200, 0), 2)
        cv2.line(f, (sx, sy), (ex, ey), (0, 255, 255), 3)
        cv2.rectangle(f, (0, 0), (w, 30), (0, 0, 0), -1)
        cv2.putText(f, f"VACH in {line.in_count} out {line.out_count} total {line.in_count+line.out_count}"
                       f"  |  VUNG {z_cur} (dinh {z_peak})  |  tong {len(seen)}",
                    (6, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        vw.write(f); i += 1
    vw.release(); cap.release()
    return dict(frames=i, tracks=len(seen), by_class=by_class,
                line=dict(in_=line.in_count, out=line.out_count, total=line.in_count + line.out_count),
                zone=dict(peak=z_peak))
print("✅ Hàm gán nhãn + đếm vạch + đếm vùng sẵn sàng.")

## 4) 2 BÀI × 2 VIDEO + vạch/vùng (theo %)
🚶 NGƯỜI: **subway** (người đi qua SẢNH ga — to, rõ) + **people-walking** (lối đi bộ).
🚗 PHƯƠNG TIỆN: **vehicles-2** (giao lộ) + **vehicles** (cao tốc). Toạ độ vạch/vùng đã
tinh chỉnh sẵn; đổi lại nếu cần cho khớp video.

In [ ]:
def dl(name):
    p = os.path.join(WORK, name)
    if not os.path.exists(p):
        os.system(f'wget -q "https://media.roboflow.com/supervision/video-examples/{name}" -O "{p}"')
    return p if os.path.exists(p) and os.path.getsize(p) > 100000 else None

TASKS = [
    # 🚶 NGƯỜI — video người ĐI TRONG SẢNH / lối đi (ngang tầm → YOLO bắt tốt)
    dict(name="NGUOI-sanh-ga", prompt="person", path=dl("subway.mp4"),
         line=[59.6, 16.8, 31.6, 90.0],                          # vạch CHÉO theo luồng qua sảnh
         zone=[[49.5, 17.8], [85.7, 17.8], [85.7, 92.4], [49.5, 92.4]]),
    dict(name="NGUOI-loi-di", prompt="person", path=dl("people-walking.mp4"),
         line=[0, 62, 100, 62],                                  # vạch ngang cắt lối đi
         zone=[[5, 45], [95, 45], [95, 92], [5, 92]]),
    # 🚗 PHƯƠNG TIỆN
    dict(name="XE-giao-lo", prompt="vehicle", path=dl("vehicles-2.mp4"),
         line=[3.5, 93.6, 93.2, 88.8],
         zone=[[15, 35], [85, 35], [85, 92], [15, 92]]),
    dict(name="XE-cao-toc", prompt="vehicle", path=dl("vehicles.mp4"),
         line=[3.4, 90.3, 98.4, 82.9],
         zone=[[5, 50], [98, 50], [98, 96], [5, 96]]),
]
for t in TASKS:
    print(("OK  " if t["path"] else "MISSING  "), t["name"], "→", t["prompt"], "|", t["path"])

## 5) ▶️ Chạy 2 bài → video (màu/nhãn/quỹ đạo + vạch + vùng) + ảnh

In [ ]:
OUTDIR = os.path.join(WORK, "label_trace_out"); os.makedirs(OUTDIR, exist_ok=True)
results = []
for t in TASKS:
    if not t["path"]:
        print("bỏ", t["name"], "(thiếu video)"); continue
    mp4 = os.path.join(OUTDIR, f"{t['name'].replace(' ', '_')}_annot_count.mp4")
    print(f"\n▶ {t['name']} — gán nhãn + đếm vạch + đếm vùng…")
    r = run_task(t["name"], t["path"], t["prompt"], t["line"], t["zone"], mp4)
    by = ", ".join(f"{k}:{c}" for k, c in sorted(r["by_class"].items(), key=lambda x: -x[1]))
    print(f"   VẠCH: in {r['line']['in_']}/out {r['line']['out']}/total {r['line']['total']}"
          f"  ·  VÙNG đỉnh {r['zone']['peak']}  ·  tổng {r['tracks']} ({by})")
    cap = cv2.VideoCapture(mp4); nn = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, nn - 2)); ok, fr = cap.read(); cap.release()
    if ok:
        plt.figure(figsize=(11, 6)); plt.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        plt.title(f"{t['name']}: vạch total {r['line']['total']} · vùng đỉnh {r['zone']['peak']} · tổng {r['tracks']}")
        plt.axis("off"); plt.show()
    results.append(dict(name=t["name"], mp4=mp4, **r))

## 6) 🎥 Xem/tải video (H.264)

In [ ]:
from IPython.display import Video, display, Markdown
for r in results:
    display(Markdown(f"### {r['name']} — VẠCH total {r['line']['total']} · VÙNG đỉnh {r['zone']['peak']} · tổng {r['tracks']}"))
    h264 = r["mp4"].replace(".mp4", "_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", r["mp4"],
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", h264], check=False)
    display(Video(h264 if os.path.exists(h264) else r["mp4"], embed=True, width=760))
print("💾 Video + ảnh đã lưu ở:", OUTDIR)

### Ghi chú
- **Đếm** bằng supervision: `LineZone` (cắt vạch in/out) + `PolygonZone` (trong vùng) trên
  track của `ByteTrack`. **tổng** = số vật khác nhau (track).
- **Màu theo LỚP** (`ColorLookup.CLASS`): car/truck/bus mỗi loại 1 màu; person tất cả 1 màu.
- **Vì sao đếm người tốt hơn?** Bỏ `market-square` (quay TỪ TRÊN, người ~vài chục px → YOLO bỏ sót).
  Dùng `subway`/`people-walking`: người ngang tầm, TO → YOLO bắt gần đủ. Muốn bắt kỹ hơn nữa:
  `os.environ['YOLO_AUGMENT']='1'` hoặc `os.environ['YOLO_IMGSZ']='1536'` (Cell 2, chậm hơn).
- **Đếm VẠCH ra 0?** vạch lệch luồng đi — sửa `line` (%) ở Cell 4 cho cắt ngang chỗ người/xe qua.
  Muốn mỗi vật 1 màu riêng (theo track): đổi `ColorLookup.CLASS`→`TRACK` (Cell 2).